# Use case: daily sediment load (time series)

Predict the suspended sediment load `St` on day *t* from discharge `Qt`, `Qt-1` and the previous
day's load `St-1`, **with prediction intervals at 90 % target coverage**.

| | |
|---|---|
| Data | 1095 training days (3 years), 365 test days (the following year) |
| Target | `St`. Heavy-tailed: the training years contain floods up to 84 400; the test year peaks at 7 400 |
| Setting | `time_ordered=True`, so nothing is shuffled and calibration uses the end of the training period |
| Compute | about 15 minutes on a Kaggle CPU notebook |

This notebook reproduces the Kaggle run documented in `docs/use-cases/sediment-load.md`.

## 1. Install

With internet access:

```bash
pip install "reguq[boosting,gpboost,ngboost,qrf,treeffuser,tuning,report] @ git+https://github.com/DaneshSelwal/reguq"
```

See `docs/guide/installation.md` for offline installs (e.g. Kaggle accounts without internet).

In [ ]:
import sys, time

import numpy as np
import pandas as pd

import reguq

print("reguq", reguq.__version__, "| python", sys.version.split()[0])

## 2. Data

Set `DATA_DIR` to a local folder containing `train.csv` and `test.csv`. By default the files are read from the public analysis repository.

In [ ]:
DATA_DIR = (
    "https://raw.githubusercontent.com/DaneshSelwal/Regression_Uncertainty_Quantification_Analysis/"
    "version1.2/examples/sediment_load_uncertainty_analysis/data"
)
TRAIN, TEST = f"{DATA_DIR}/train.csv", f"{DATA_DIR}/test.csv"

train, test = pd.read_csv(TRAIN), pd.read_csv(TEST)
print(train.shape, test.shape, list(train.columns))
pd.DataFrame({"train": train["St"].describe(), "test": test["St"].describe()}).round(1)

## 3. One method by hand

Before running everything, here is what the pipeline does for a single method. We fit
split conformal and ACI on one model, then evaluate both on the test year.

In [ ]:
from lightgbm import LGBMRegressor

from reguq.conformal import OnlineConformalRegressor, SplitConformalRegressor
from reguq.metrics import interval_report

X_tr, y_tr = train[["Qt", "Qt-1", "St-1"]], train["St"].to_numpy(float)
X_te, y_te = test[["Qt", "Qt-1", "St-1"]], test["St"].to_numpy(float)
base = LGBMRegressor(n_estimators=300, learning_rate=0.05, verbose=-1, random_state=0)

split = SplitConformalRegressor(base, calib_size=0.25, time_ordered=True).fit(X_tr, y_tr)
aci = OnlineConformalRegressor(base, method="aci", alpha=0.1).fit(X_tr, y_tr)

pd.DataFrame({
    "split (static)": interval_report(y_te, split.predict_interval(X_te, 0.1), alpha=0.1),
    "ACI (online)": interval_report(y_te, aci.predict_interval_online(X_te, y_te), alpha=0.1),
}).T.round(3)

Split conformal is calibrated on the end of the training period, which had a different flood
regime, so it can miss the 90 % target on the new year. ACI updates its level after each observed
day, and its long-run coverage approaches 90 % for any sequence.

## 4. Full pipeline: 6 tuned models × 14 methods + 5 model-free methods

In [ ]:
from reguq.pipeline import PipelineConfig, run_pipeline

cfg = PipelineConfig(
    train_path=TRAIN, test_path=TEST, target="St",
    time_ordered=True, alpha=0.1, calib_size=0.25, random_state=0,
    models=["lightgbm", "xgboost", "catboost", "gradientboosting", "randomforest", "gpboost"],
    methods=["split", "normalized", "cqr", "cv_plus", "jab", "nexcp", "aci", "dtaci", "sf_ogd",
             "saocp", "nexcp_online", "acp", "multi_quantile", "ibug"],
    model_free_methods=["ngboost", "qrf", "hcm", "treeffuser"],
    tuning={"n_trials": 40, "n_splits": 5, "sampler": "tpe", "pruner": "median"},
    method_params={"jab": {"n_bootstraps": 20}, "cv_plus": {"n_splits": 5}},
    output_dir="results_sediment_load", export_excel=True, export_plots=True,
)
t0 = time.time()
res = run_pipeline(cfg)
print(f"{time.time() - t0:.0f} s")

Hyper-parameters were tuned with forward-chaining CV on the **training years only**:

In [ ]:
pd.DataFrame(res.tuning).T

## 5. Results

Rank by the **interval score** (lower is better; it rewards narrow intervals and penalises misses by 2/α × distance). Look at coverage next to it: rows far below 0.90 are not valid on this data, however narrow they are.

In [ ]:
cols = ["method", "model", "status", "coverage", "mean_width", "interval_score", "rmse", "r2", "crps"]
m = res.metrics
m[m.status == "ok"][[c for c in cols if c in m]].sort_values("interval_score").round(3).head(20)

In [ ]:
(m[m.status == "ok"].groupby("method")
   .agg(coverage=("coverage", "mean"), cov_min=("coverage", "min"),
        interval_score=("interval_score", "median"), width=("mean_width", "median"))
   .sort_values("interval_score").round(3))

Combinations that cannot exist are reported, never replaced. For example, random forest has no quantile objective, and IBUG cannot read GPBoost's leaves:

In [ ]:
m[m.status != 'ok'][['method', 'model', 'status']]

## 6. Plots

In [ ]:
import matplotlib.pyplot as plt

from reguq.reporting import plot_coverage_width, plot_intervals

plot_coverage_width(m, alpha=0.1)
p = res.predictions["dtaci__lightgbm"]
plot_intervals(p.y_true, p.y_pred, p.lower, p.upper, title="DtACI + LightGBM, test year")
plt.show()

## 7. Take-aways

* **Online methods (ACI, DtACI, SF-OGD) reach the 90 % target for every model.** This is what their
  distribution-free long-run guarantee predicts under the regime change between the training and test years.
* Static methods (split, normalized, CQR) give narrower intervals but **undercover**, because they are
  calibrated on a flood regime the test year does not share.
* CV+, J+aB, SAOCP and ACP are conservative here. ACP's mean-based (e-value) interval is wide by construction.
* The errors are dominated by a few flood days. A log-transformed target is a natural next experiment.

Method details and guarantees: `docs/methods/`. Known issues with the earlier notebooks: `docs/audit/notebooks-v1.2.md` (D5, D6).